# IT3091 Machine Learning - Group 2026-AI-16
## Sri Lankan Vegetable Yield + Next-Month Wholesale Price Prediction

**Track:** Industry Explorer  
**ML task:** Supervised multi-output regression  
**Output 1:** Crop yield (t/ha)  
**Output 2:** Next-month wholesale price (Rs/kg)  
**Decision lens:** Farmer / agricultural production planning.

### Why this notebook uses a national Crop + Season + Year row
Your price file contains a Colombo/suburbs wholesale price series for each commodity, not a separate price for every production district. Repeating the same price target across all districts would create artificial duplicate target labels. Therefore this starter uses:

**One modelling row = Crop + Season + Year**

- Production: `National Total` rows only.
- Weather: all district weather is combined into a national seasonal climate summary.
- Price: wholesale monthly price series derived from weekly data.

This is a defensible common unit for one multi-output model. If your lecturer explicitly requires district-level price prediction, obtain district-specific price data before changing this design.

## Cell 1 - Install/import libraries
Google Colab normally already contains most of these packages. Run this cell first.

In [4]:
!pip -q install openpyxl joblib

import os, re, json, hashlib, zipfile, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.multioutput import MultiOutputRegressor
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
RANDOM_STATE = 42
print('Libraries ready')

Libraries ready


## Cell 2 - GitHub repository and dataset folders

In [ ]:
# GitHub repository paths
REPO_PATH = Path('/content/IT3091---Machine-Learning-Assignment')

if not REPO_PATH.exists():
    raise FileNotFoundError(
        "Clone the GitHub repository first:\n"
        "!git clone https://github.com/DTD-Wijesinghe/IT3091---Machine-Learning-Assignment.git"
    )

os.chdir(REPO_PATH)

RAW_PATH = REPO_PATH / 'Raw Datasets'
PREPROCESSED_PATH = REPO_PATH / 'Preprocessed Datasets'

if not RAW_PATH.exists():
    raise FileNotFoundError(f"Raw Datasets folder not found: {RAW_PATH}")

PREPROCESSED_PATH.mkdir(parents=True, exist_ok=True)

print('Repository:', REPO_PATH)
print('Raw datasets:', RAW_PATH)
print('Preprocessed datasets:', PREPROCESSED_PATH)

## Cell 3 - Raw dataset file paths

In [ ]:
BASE = RAW_PATH
PRODUCTION_FILE = BASE / 'researchData.xlsx'
WEATHER_FILE = BASE / 'weatherData.csv'
PRICE_FILE = BASE / 'Weekly Average Retail Prices of Vegi - 2012 to 2025.xlsx'
LOCATION_FILE = BASE / 'locationData.csv'
DAILY_PRICE_FILE = BASE / '2018-2025 Daily Wholesale Prices(Colombo).xlsx'
WEATHER_ZIP = BASE / 'archive (2).zip'

if not LOCATION_FILE.exists() and WEATHER_ZIP.exists():
    with zipfile.ZipFile(WEATHER_ZIP, 'r') as z:
        z.extract('locationData.csv', BASE)
    print('Extracted locationData.csv from archive (2).zip')

required = [PRODUCTION_FILE, WEATHER_FILE, PRICE_FILE, LOCATION_FILE]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(f'Missing required file(s): {missing}')

print('All required raw files found')

## Cell 4 - Reproducibility evidence: file fingerprints
Keep this output in the notebook/report. A SHA-256 fingerprint proves which exact file version was used.

In [10]:
def sha256_file(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

fingerprints = []
for p in [PRODUCTION_FILE, WEATHER_FILE, PRICE_FILE, LOCATION_FILE, DAILY_PRICE_FILE]:
    if p.exists():
        fingerprints.append({
            'file': p.name,
            'bytes': p.stat().st_size,
            'sha256': sha256_file(p)
        })

fingerprints_df = pd.DataFrame(fingerprints)
display(fingerprints_df)
fingerprints_df.to_csv('dataset_fingerprints.csv', index=False)

,file,bytes,sha256
0,researchData.xlsx,2776631,4b2f922b03d93c0d7a46b68aba1546db9278767a4ab910...
1,weatherData.csv,15796522,cc75e8c1cdc82d5ee0f2e75e80490fecc050247619455b...
2,Weekly Average Retail Prices of Vegi - 2012 to...,241424,ab5303299e29ccbbcd495cea646fbe00f06b88adcd0c5a...
3,locationData.csv,1657,912c02712a422632650f510ce00bb4a5754aa823384fd5...
4,2018-2025 Daily Wholesale Prices(Colombo).xlsx,250996,a48cc494eeb10258331de414ae0e4323e7cc59d5066f43...


## Cell 5 - Record dataset sources
Replace any placeholder with the **exact page/download link your group used**. Keep these links in the final report.

Useful provenance pages:
- Weather dataset: https://www.kaggle.com/datasets/tharindumadhusanka9/sri-lanka-weather-data-for-all-districts
- HARTI weekly prices: https://www.harti.gov.lk/weekly-price.php
- Department of Census and Statistics agriculture portal: https://www.statistics.gov.lk/Agriculture

In [11]:
DATA_SOURCES = {
    'production': 'PASTE_EXACT_DCS_OR_GOVERNMENT_DOWNLOAD_URL_USED_BY_GROUP',
    'weather': 'https://www.kaggle.com/datasets/tharindumadhusanka9/sri-lanka-weather-data-for-all-districts',
    'weekly_price': 'https://www.harti.gov.lk/weekly-price.php',
    'daily_price_supporting': 'PASTE_EXACT_SOURCE_URL_IF_USED'
}
print(json.dumps(DATA_SOURCES, indent=2))

{
  "production": "PASTE_EXACT_DCS_OR_GOVERNMENT_DOWNLOAD_URL_USED_BY_GROUP",
  "weather": "https://www.kaggle.com/datasets/tharindumadhusanka9/sri-lanka-weather-data-for-all-districts",
  "weekly_price": "https://www.harti.gov.lk/weekly-price.php",
  "daily_price_supporting": "PASTE_EXACT_SOURCE_URL_IF_USED"
}


# Part A - Production data
## Cell 6 - Load and inspect `researchData.xlsx`
Expected main fields: District, Season, CropCategory, Crop, Year, Extent, Production.

In [12]:
prod_raw = pd.read_excel(PRODUCTION_FILE)
prod_raw = prod_raw.loc[:, ~prod_raw.columns.astype(str).str.startswith('Unnamed')]
prod_raw = prod_raw.drop(columns=['index'], errors='ignore')

print('Shape:', prod_raw.shape)
display(prod_raw.head())
print('\nColumns:', list(prod_raw.columns))
print('\nMissing values:')
display(prod_raw.isna().sum().sort_values(ascending=False).to_frame('missing'))
print('\nCrop categories:')
print(sorted(prod_raw['CropCategory'].dropna().astype(str).unique()))

Shape: (94755, 7)


,District,Season,CropCategory,Crop,Year,Extent,Production
0,National Total,Yala,Cereals,Kurakkan,2001,650.0,422.0
1,National Total,Maha,Cereals,Kurakkan,2001,"4,986.0","3,775.0"
2,National Total,Total,Cereals,Kurakkan,2001,"5,636.0","4,197.0"
3,National Total,Yala,Cereals,Kurakkan,2002,647.0,408.0
4,National Total,Maha,Cereals,Kurakkan,2002,"4,830.0","3,663.0"



Columns: ['District', 'Season', 'CropCategory', 'Crop', 'Year', 'Extent', 'Production']

Missing values:


,missing
Production,17016
Extent,16644
District,0
CropCategory,0
Season,0
Year,0
Crop,0



Crop categories:
['Cereals', 'Fruits', 'Leaves', 'Low Country Vegetable', 'Minor Export', 'Oil Seeds', 'Other', 'Other Perennial', 'Pulses', 'Roots and Tubers', 'Up Country Vegetable']


## Cell 7 - Clean production data and create the Yield target
Rules used:
- Keep `National Total` because the multi-output price target is not district-specific.
- Keep only `Maha` and `Yala`; remove `Total` because it duplicates the two seasons.
- Focus on **Up Country Vegetable + Low Country Vegetable** so the production crops match the vegetable price file cleanly.
- Use 2012 onward.
- `Yield_t_ha = Production / Extent`.
- `Production` is **never** used later as an input feature.

In [13]:
prod = prod_raw.copy()

# Text cleanup
for c in ['District','Season','CropCategory','Crop']:
    prod[c] = prod[c].astype('string').str.strip()

# Numeric cleanup (handles values like "4,986.0")
for c in ['Year','Extent','Production']:
    prod[c] = pd.to_numeric(
        prod[c].astype(str).str.replace(',', '', regex=False).str.strip(),
        errors='coerce'
    )

VEG_CATEGORIES = ['Up Country Vegetable', 'Low Country Vegetable']
prod = prod[
    (prod['District'].eq('National Total')) &
    (prod['Season'].isin(['Maha','Yala'])) &
    (prod['CropCategory'].isin(VEG_CATEGORIES)) &
    (prod['Year'].between(2012, 2025))
].copy()

# Yield is undefined when Extent <= 0.
prod = prod[(prod['Extent'] > 0) & (prod['Production'].notna()) & (prod['Production'] >= 0)].copy()
prod['Year'] = prod['Year'].astype(int)
prod['Yield_t_ha'] = prod['Production'] / prod['Extent']
prod = prod.replace([np.inf, -np.inf], np.nan).dropna(subset=['Yield_t_ha'])

print('Clean production shape:', prod.shape)
print('Year range:', prod['Year'].min(), '-', prod['Year'].max())
print('Crops:', sorted(prod['Crop'].unique()))
display(prod.head())

Clean production shape: (699, 8)
Year range: 2012 - 2025
Crops: ['Ash Plantain', 'Ash Pumpkin', 'Bandakka', 'Beans', 'Beetroot', 'Bitter Gourd', 'Brinjals', 'Cabbage', 'Capsicum', 'Carrot', 'Coli Flower', 'Cucumber', 'Drumstics', 'Egg Plant', 'Gherkin', 'Kekiri', 'Knolkhol', 'Leeks', 'Long Bean', 'Luffa', 'Raddish', 'Red Pumpkin', 'Snake Gourd', 'Thumba Karavila', 'Tomatoes', 'Vatana', 'Winged Bean']


,District,Season,CropCategory,Crop,Year,Extent,Production,Yield_t_ha
30006,National Total,Yala,Low Country Vegetable,Luffa,2012,1861.0,17053.0,9.163353
30007,National Total,Maha,Low Country Vegetable,Luffa,2012,2840.0,25733.0,9.060915
30009,National Total,Yala,Low Country Vegetable,Luffa,2013,1985.0,17801.0,8.967758
30010,National Total,Maha,Low Country Vegetable,Luffa,2013,3005.0,28598.0,9.516805
30012,National Total,Yala,Low Country Vegetable,Luffa,2014,1881.0,16066.0,8.541201


## Cell 8 - Production data-quality checks
Do not automatically delete all outliers. Agricultural extremes can be real. First flag and inspect them, then record your decision in the EDA/preprocessing logs.

In [16]:
def iqr_flag_series(s):
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return (s < lower) | (s > upper)


prod['Yield_IQR_Flag'] = (
    prod.groupby('Crop')['Yield_t_ha']
        .transform(iqr_flag_series)
        .fillna(False)
        .astype(bool)
)

print('Rows flagged for review:', int(prod['Yield_IQR_Flag'].sum()))

display(
    prod[prod['Yield_IQR_Flag']]
    .sort_values('Yield_t_ha', ascending=False)
    .head(20)
)

Rows flagged for review: 35


,District,Season,CropCategory,Crop,Year,Extent,Production,Yield_t_ha,Yield_IQR_Flag
49774,National Total,Maha,Up Country Vegetable,Leeks,2025,891.8,45804.2,51.361516,True
49771,National Total,Maha,Up Country Vegetable,Leeks,2024,1436.9,69673.1,48.488482,True
49740,National Total,Yala,Up Country Vegetable,Leeks,2014,430.0,18354.0,42.683721,True
47767,National Total,Maha,Up Country Vegetable,Raddish,2024,1368.1,40159.0,29.353848,True
72646,National Total,Maha,Low Country Vegetable,Vatana,2016,262.0,7400.0,28.244275,True
86679,National Total,Yala,Up Country Vegetable,Knolkhol,2019,353.0,8584.0,24.317280,True
72645,National Total,Yala,Low Country Vegetable,Vatana,2016,434.0,9770.0,22.511521,True
35716,National Total,Maha,Low Country Vegetable,Snake Gourd,2015,1530.0,29491.0,19.275163,True
39747,National Total,Yala,Low Country Vegetable,Cucumber,2023,1456.0,26602.8,18.271154,True
33724,National Total,Maha,Low Country Vegetable,Bitter Gourd,2019,1431.0,25004.0,17.473096,True


# Part B - Weather data
## Cell 9 - Load weather + location mapping
The weather file contains `location_id`. Never guess the district. Join it to `locationData.csv` first.

In [17]:
weather = pd.read_csv(WEATHER_FILE)
locations = pd.read_csv(LOCATION_FILE)

# Remove exported index columns if present
weather = weather.loc[:, ~weather.columns.astype(str).str.lower().isin(['index','unnamed: 0'])]
locations = locations.loc[:, ~locations.columns.astype(str).str.lower().isin(['index','unnamed: 0'])]

print('Weather:', weather.shape)
print('Locations:', locations.shape)
display(locations[['location_id','city_name']].head(10))

weather = weather.merge(
    locations[['location_id','city_name']],
    on='location_id', how='left', validate='many_to_one'
)

if weather['city_name'].isna().any():
    raise ValueError('Some weather location_id values did not map to locationData.csv')

weather['date'] = pd.to_datetime(weather['date'], errors='coerce')
weather = weather.dropna(subset=['date']).copy()
print('Weather date range:', weather['date'].min(), 'to', weather['date'].max())

Weather: (142371, 21)
Locations: (27, 8)


,location_id,city_name
0,0,Colombo
1,1,Gampaha
2,2,Kalutara
3,3,Kandy
4,4,Matale
5,5,Nuwara Eliya
6,6,Galle
7,7,Matara
8,8,Hambantota
9,9,Jaffna


Weather date range: 2010-01-01 00:00:00 to 2024-06-08 00:00:00


## Cell 10 - Create agricultural Season + SeasonYear
The weather dataset's published analysis uses **Sep-Mar** and **Apr-Aug** seasonal windows. We use:
- **Maha:** September-March
- **Yala:** April-August

For Maha, January-March are assigned to the previous season-start year. Example: January 2023 belongs to `Maha 2022`.

**Important:** verify that this `SeasonYear` interpretation matches the original production-data source and record that confirmation in your decision log.

In [18]:
def season_and_year(dt):
    m, y = dt.month, dt.year
    if m in [4,5,6,7,8]:
        return pd.Series(['Yala', y])
    if m in [9,10,11,12]:
        return pd.Series(['Maha', y])
    if m in [1,2,3]:
        return pd.Series(['Maha', y-1])
    return pd.Series([pd.NA, pd.NA])

weather[['Season','Year']] = weather['date'].apply(season_and_year)
weather['Year'] = pd.to_numeric(weather['Year'], errors='coerce').astype('Int64')
weather = weather[weather['Year'].between(2012, 2023)].copy()

print(weather[['date','Season','Year']].head())
print('Coverage by season/year:')
display(weather.groupby(['Year','Season']).size().unstack(fill_value=0).tail())

          date Season  Year
821 2012-04-01   Yala  2012
822 2012-04-02   Yala  2012
823 2012-04-03   Yala  2012
824 2012-04-04   Yala  2012
825 2012-04-05   Yala  2012
Coverage by season/year:


Season,Maha,Yala
Year,,
2019,5751,4131
2020,5724,4131
2021,5724,4131
2022,5724,4131
2023,5751,4131


## Cell 11 - National seasonal weather features
First average each weather variable across available districts for each day; then summarize the daily national averages across the season. This avoids making rainfall totals grow simply because there are many districts.

In [19]:
WEATHER_COLS = {
    'temperature_2m_mean (°C)': 'temp',
    'precipitation_sum (mm)': 'precip',
    'wind_speed_10m_max (km/h)': 'wind',
    'shortwave_radiation_sum (MJ/m²)': 'solar',
    'et0_fao_evapotranspiration (mm)': 'et0'
}

for c in WEATHER_COLS:
    weather[c] = pd.to_numeric(weather[c], errors='coerce')

# Daily national mean across locations
weather_daily_nat = (
    weather.groupby(['date','Season','Year'], as_index=False)[list(WEATHER_COLS)]
           .mean()
)

weather_season = (
    weather_daily_nat.groupby(['Year','Season'], as_index=False)
    .agg(
        SeasonMeanTemp_C=('temperature_2m_mean (°C)', 'mean'),
        SeasonTotalPrecip_mm=('precipitation_sum (mm)', 'sum'),
        SeasonMeanWind_kmh=('wind_speed_10m_max (km/h)', 'mean'),
        SeasonMeanSolar_MJm2=('shortwave_radiation_sum (MJ/m²)', 'mean'),
        SeasonTotalET0_mm=('et0_fao_evapotranspiration (mm)', 'sum'),
        WeatherDays=('date','nunique')
    )
)

# Coverage check: incomplete seasons should be reviewed.
print('Seasonal weather summary:')
display(weather_season.tail(10))

Seasonal weather summary:


,Year,Season,SeasonMeanTemp_C,SeasonTotalPrecip_mm,SeasonMeanWind_kmh,SeasonMeanSolar_MJm2,SeasonTotalET0_mm,WeatherDays
14,2019,Maha,25.286281,1221.614815,15.525995,18.702238,844.108889,213
15,2019,Yala,27.137085,602.092593,19.153232,19.594607,701.223333,153
16,2020,Maha,25.094392,1391.496296,16.164832,17.787799,796.865926,212
17,2020,Yala,26.812709,690.314815,17.526846,19.334619,661.879259,153
18,2021,Maha,25.023008,1186.703704,15.790391,18.831791,835.437037,212
19,2021,Yala,26.357081,850.007407,18.162890,19.374212,656.260000,153
20,2022,Maha,24.578616,1171.896296,15.951013,18.837386,827.268889,212
21,2022,Yala,26.052554,857.596296,18.655604,19.427855,651.735556,153
22,2023,Maha,25.355208,1769.181481,15.518362,18.635681,839.443704,213
23,2023,Yala,26.854829,588.022222,18.153813,20.775805,717.528519,153


## Save Member 2 preprocessed weather dataset

In [ ]:
weather_output = PREPROCESSED_PATH / '02_weather_preprocessed.csv'
weather_season.to_csv(weather_output, index=False)
print('Saved:', weather_output)